# 03 — Convolution from scratch

## Roadmap
1. Definition of 2-D convolution
2. Implement `conv2d` in 30 lines of NumPy
3. Hand-crafted kernels (edge, blur, sharpen) — same operator, different weights
4. Padding, stride, dilation — the three knobs
5. **Output-size formula** with worked examples
6. Pooling (max, average) and why it exists
7. The receptive field


In [ ]:
# === Bootstrap (safe to re-run) ===
# Installs minimal deps and downloads tiny CIFAR-10 subset for any notebook
# that needs it. The from-scratch notebooks deliberately avoid importing the
# project package so they run anywhere (Colab, plain Python, Jupyter).
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "matplotlib", "pillow", "torchvision"], check=True)
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print("numpy", np.__version__)


## 1. Definition

2-D cross-correlation of image $I$ with kernel $K$ at position $(y, x)$:

$$(I * K)(y, x) = \sum_{i=0}^{K_h-1} \sum_{j=0}^{K_w-1} I(y+i, x+j) \cdot K(i, j)$$

"Slide the kernel over the image, multiply element-wise, sum." Repeat at every position.

(Pedantic note: math literature calls this *cross-correlation*; deep-learning practice calls it *convolution*. We follow the deep-learning convention.)


In [ ]:
def conv2d(img, kernel, padding=0, stride=1):
    """Single-channel 2-D convolution from scratch.

    img:    (H, W) float array
    kernel: (kh, kw) float array
    Returns: (Hout, Wout) float array
    """
    if padding > 0:
        img = np.pad(img, padding, mode="constant", constant_values=0)
    H, W = img.shape
    kh, kw = kernel.shape
    Hout = (H - kh) // stride + 1
    Wout = (W - kw) // stride + 1
    out = np.zeros((Hout, Wout), dtype=np.float32)
    for y in range(Hout):
        for x in range(Wout):
            patch = img[y*stride:y*stride+kh, x*stride:x*stride+kw]
            out[y, x] = (patch * kernel).sum()
    return out

print("conv2d ready")


## 2. Hand-crafted kernels

The same `conv2d` does completely different things depending on the kernel values.


In [ ]:
# Build a tiny test image: vertical line down the middle
img = np.zeros((9, 9), dtype=np.float32)
img[:, 4] = 1.0

sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=np.float32)
blur    = np.ones((3, 3), dtype=np.float32) / 9.0
sharpen = np.array([[ 0, -1,  0],
                    [-1,  5, -1],
                    [ 0, -1,  0]], dtype=np.float32)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, (k, t) in zip(axes, [(None, "input"),
                              (sobel_x, "sobel-x (vertical edge)"),
                              (blur, "blur"),
                              (sharpen, "sharpen")]):
    if k is None:
        ax.imshow(img, cmap="gray"); ax.set_title(t)
    else:
        ax.imshow(conv2d(img, k, padding=1), cmap="RdBu"); ax.set_title(t)
    ax.axis("off")
plt.show()


**The whole point of training a CNN:** instead of you hand-picking these kernel values, the network *learns* them by gradient descent. Early layers tend to learn edge / color detectors that look very much like Sobel.

## 3. Padding, stride, dilation

| Knob | Effect | Use |
|---|---|---|
| **Padding** P | Adds P pixels of zeros around the input. Lets you preserve spatial size after conv. | "Same padding": $P = (K-1)/2$ keeps $H_\text{out} = H_\text{in}$ when stride 1. |
| **Stride** S | Move the kernel S steps each time. Downsamples by factor S. | Used instead of pooling in modern CNNs (ResNet uses stride-2 convs). |
| **Dilation** D | Spread kernel cells D pixels apart. Larger receptive field, same params. | Semantic segmentation (DeepLab). |

## 4. Output-size formula

$$H_\text{out} = \left\lfloor \frac{H_\text{in} + 2P - K}{S} \right\rfloor + 1$$


In [ ]:
def conv_output_size(H, K, P, S):
    return (H + 2*P - K) // S + 1

for (H, K, P, S, expected) in [
    (32, 3, 1, 1, 32),    # same conv
    (32, 3, 0, 1, 30),    # no padding
    (32, 3, 1, 2, 16),    # strided conv = downsample
    (224, 7, 3, 2, 112),  # ResNet50's first conv
]:
    got = conv_output_size(H, K, P, S)
    print(f"H={H} K={K} P={P} S={S} → {got}  ({'OK' if got==expected else f'expected {expected}'})")


## 5. Pooling

A pooling layer downsamples by taking the max (or average) over each KxK window. No learnable parameters. Two reasons it exists:

1. **Translation invariance** — small shifts of an object don't change the output.
2. **Spatial reduction** — fewer pixels = cheaper subsequent layers.

Modern CNNs (ResNet, EfficientNet) often use stride-2 convs *instead* of pooling, but it's still everywhere.


In [ ]:
def max_pool(img, k=2, s=2):
    H, W = img.shape
    Hout = (H - k) // s + 1
    Wout = (W - k) // s + 1
    out = np.zeros((Hout, Wout), dtype=np.float32)
    for y in range(Hout):
        for x in range(Wout):
            out[y, x] = img[y*s:y*s+k, x*s:x*s+k].max()
    return out

x = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10,11,12],
              [13,14,15,16]], dtype=np.float32)
print("input:\n", x)
print("\nmax_pool 2x2, stride 2:\n", max_pool(x, k=2, s=2))


## 6. Receptive field — what one output pixel "sees"

After $L$ stacked 3×3 convs (stride 1), one output pixel depends on a $(2L+1) \times (2L+1)$ region of the input.

After 1 conv: 3×3. After 2 convs: 5×5. After 10 convs: 21×21.

This is why **depth helps**: deep stacks of small kernels see a large region without the parameter cost of one giant kernel.

**Next:** stack convs into a tiny CNN, train it on CIFAR, then add the residual trick.
